# Entrenar el U-Net de copas (RGB + CHM) en Google Colab

Este notebook:
1. Clona el repo y instala dependencias (activar GPU antes: `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`).
2. Monta tu Google Drive para leer los rasters (RGB, CHM, labels) y guardar el modelo entrenado.
3. Tilea los rasters en parches de entrenamiento.
4. Entrena y guarda `checkpoints/best.pth`.
5. Copia `best.pth` a Drive (o lo descarga directo) para que no se pierda al cerrarse la sesión de Colab.

In [ ]:
!nvidia-smi  # confirmar que hay GPU asignada (si falla, activarla en el menu de entorno de ejecucion)

## 1. Clonar el repo e instalar dependencias

In [ ]:
BRANCH = "claude/unet-tree-canopy-detection-88jcff"  # cambiar a "main" una vez mergeado

!git clone --branch $BRANCH https://github.com/KeylaFrancoH/tree_crown_detection.git
%cd tree_crown_detection

In [ ]:
# rasterio/geopandas ya suelen venir con wheels binarias en Colab, pero por las dudas
# se instalan primero por separado para evitar conflictos de resolución de dependencias.
!pip install -q rasterio geopandas shapely
!pip install -q -r requirements.txt

## 2. Montar Google Drive con tus datos

Subi antes a tu Drive los 3 rasters (misma grilla): imagen RGB, CHM/nDSM y el raster de copas segmentadas (0=fondo, cada copa con un ID entero distinto). Ajusta las rutas de abajo a donde los hayas guardado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Editar estas 3 rutas segun donde subiste tus archivos en Drive.
# Para tener train y val, lo mas simple es usar 2 sitios (o 2 subareas) distintos;
# aca se usa el mismo sitio para ambos solo como ejemplo minimo para probar el pipeline.
RGB_TRAIN = "/content/drive/MyDrive/tree_crowns/rgb_sitio1.tif"
CHM_TRAIN = "/content/drive/MyDrive/tree_crowns/chm_sitio1.tif"
LABELS_TRAIN = "/content/drive/MyDrive/tree_crowns/labels_sitio1.tif"

RGB_VAL = "/content/drive/MyDrive/tree_crowns/rgb_sitio2.tif"
CHM_VAL = "/content/drive/MyDrive/tree_crowns/chm_sitio2.tif"
LABELS_VAL = "/content/drive/MyDrive/tree_crowns/labels_sitio2.tif"

## 3. Tilear (recortar en parches) los rasters

Se guardan en el disco local de Colab (`/content/...`), no en Drive, porque leer/escribir muchos archivos chicos en Drive es lento.

In [ ]:
!python scripts/prepare_tiles.py \
  --rgb "$RGB_TRAIN" --chm "$CHM_TRAIN" --labels "$LABELS_TRAIN" \
  --out /content/data/train --tile-size 256 --overlap 32 --min-valid-fraction 0.1

!python scripts/prepare_tiles.py \
  --rgb "$RGB_VAL" --chm "$CHM_VAL" --labels "$LABELS_VAL" \
  --out /content/data/val --tile-size 256 --overlap 32 --min-valid-fraction 0.1

## 4. Config de entrenamiento

Se parte de `configs/default.yaml` y se sobreescriben solo las rutas (a `/content/data/...`) y el `checkpoint_dir`. Ajusta `chm_max` a la altura maxima real de tus arboles, y `epochs`/`batch_size` segun la GPU que te haya tocado (T4 gratis: batch_size 8-16 suele andar bien con base_channels=32 y tiles de 256px).

In [ ]:
import yaml

with open("configs/default.yaml") as f:
    config = yaml.safe_load(f)

config["data"]["train_dir"] = "/content/data/train"
config["data"]["val_dir"] = "/content/data/val"
config["data"]["chm_max"] = 40.0  # <-- ajustar a la altura maxima real de tus copas (metros)
config["training"]["epochs"] = 50
config["training"]["batch_size"] = 8
config["training"]["checkpoint_dir"] = "checkpoints"

with open("configs/colab.yaml", "w") as f:
    yaml.safe_dump(config, f)

print(open("configs/colab.yaml").read())

## 5. Entrenar

Guarda `checkpoints/last.pth` (ultima epoca) y `checkpoints/best.pth` (mejor IoU de validacion) a medida que entrena.

In [ ]:
!python scripts/train.py --config configs/colab.yaml

## 6. Guardar el modelo entrenado (.pth)

La sesion de Colab se borra al desconectarse, asi que copia `best.pth` a Drive (o descargalo directo) antes de cerrar.

In [ ]:
import shutil, os

os.makedirs("/content/drive/MyDrive/tree_crowns/checkpoints", exist_ok=True)
shutil.copy("checkpoints/best.pth", "/content/drive/MyDrive/tree_crowns/checkpoints/best.pth")
print("Copiado a Drive: /content/drive/MyDrive/tree_crowns/checkpoints/best.pth")

In [ ]:
# Alternativa: descarga directa al navegador en vez de (o ademas de) copiar a Drive.
from google.colab import files
files.download("checkpoints/best.pth")

## 7. (Opcional) Probar inferencia sobre un ortomosaico y ver las copas detectadas

In [ ]:
RGB_TEST = RGB_VAL
CHM_TEST = CHM_VAL

!python scripts/predict.py \
  --config configs/colab.yaml \
  --checkpoint checkpoints/best.pth \
  --rgb "$RGB_TEST" --chm "$CHM_TEST" \
  --out /content/drive/MyDrive/tree_crowns/resultados_copas.geojson